# 1. Import Library & Load Data

In [20]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

# Setup Path
BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / 'data'
INPUT_PATH = DATA_DIR / '(prod)6_emotions.csv'

# Load Dataset
df_emo = pd.read_csv(INPUT_PATH)
print(f"Total rows: {len(df_emo):,} rows")

Total rows: 38,258 rows


# 2. Mild Preprocessing 

In [21]:
def preprocess_statement(statement):
    # Lowercase & Format String
    sentence = str(statement).lower()

    # Remove Tag Reddit & URL
    sentence = re.sub(r'\[.*?\]', '', sentence) 
    sentence = re.sub(r'http\S+|www\.\S+', '', sentence) 
    sentence = re.sub(r'\@\w+', '', sentence)

    # Remove symbols
    sentence = re.sub(r'[^a-zA-Z0-9\s.,!?\']', '', sentence)

    # Remove double space if any
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    return sentence

# 3. Cleaning Execution

## GoEmotions Dataset

In [22]:
df_emo['clean_text'] = df_emo['text'].apply(preprocess_statement)

In [23]:
df_emo['clean_text'] = df_emo['clean_text'].replace('', np.nan)
df_emo.isna().sum()

text             0
emotion_label    0
clean_text       2
dtype: int64

In [24]:
before_drop = len(df_emo)
df_emo = df_emo.dropna(subset=['clean_text']).reset_index(drop=True)
after_drop = len(df_emo)

In [25]:
display(df_emo.head())

,text,emotion_label,clean_text
0,That game hurt.,Sadness,that game hurt.
1,Man I love reddit.,Love,man i love reddit.
2,So happy for [NAME]. So sad he's not here. Ima...,Sadness,so happy for . so sad he's not here. imagine t...
3,"I just came home, what the fuck is this lineup...",Love,"i just came home, what the fuck is this lineup..."
4,By far the coolest thing I've seen on this thr...,Joy,by far the coolest thing i've seen on this thr...


In [26]:
print(f"Removed noisy rows: {before_drop - after_drop:,}")
print(f"Clean rows ready for training: {after_drop:,}")

Removed noisy rows: 2
Clean rows ready for training: 38,256


In [27]:
OUTPUT_PATH = DATA_DIR / '(prod)goemotions_train.csv'

df_emo.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")